In [1]:
from data_processing import *
from kernels import *
from methods import *
from utils import *
from features import *
from data_augmentation import *
from sklearn.svm import SVC

In [2]:
# Load data
Xtr, Xte, Ytr = load_data(data_dir="../")
# Preprocess data
std_Xtr, Xtr, Xte = normalize_data(Xtr, Xte)
Xtr_train, Ytr_train, Xtr_val, Ytr_val = split_train_test(Xtr, Ytr, test_size=0.2)
X_train, Y_train = data_transformations(Xtr_train, Ytr_train, only_flip=True)
# Hog features
X_hog_train = extract_hog_features(X_train, cell_size=8, bins=9, block_size=2)
X_hog_val   = extract_hog_features(Xtr_val, cell_size=8, bins=9, block_size=2)
# Y labels
Y_train_labels = 2*one_hot_encode(Y_train, 10)-1

Data loaded successfully
Xtr.shape: (5000, 3072)
Xte.shape: (2000, 3072)
Ytr.shape: (5000,)
Before scaling:
Overall mean: 0.00025552399643960895
Overall std: 0.036931775770682115

After scaling:
Xtr mean: 0.006918811541210913
Xtr std: 0.9999999999999991
Xte mean: 0.005958882871338156
Xte std: 0.9983245277147763

Values to save for later use:
mean_Xtr = 0.00025552399643960895
std_Xtr = 0.036931775770682115


In [3]:
# Kernels 
Linear = LinearKernel()
Poly = PolynomialKernel()
RBF = RBFKernel(gamma=1)
Chi2 = ChiSquareKernel(gamma=0.2) 
# Train kernels 
K_train_linear_hog = Linear(X_hog_train, X_hog_train)
K_train_poly_hog = Poly(X_hog_train, X_hog_train)
K_train_rbf_hog = RBF(X_hog_train, X_hog_train)
K_train_chi_hog = Chi2(X_hog_train, X_hog_train, block_size=128)
# Validation kernels
K_val_linear_hog = Linear(X_hog_val, X_hog_train)
K_val_poly_hog = Poly(X_hog_val, X_hog_train)
K_val_rbf_hog = RBF(X_hog_val, X_hog_train)
K_val_chi_hog = Chi2(X_hog_val, X_hog_train, block_size=128)
# For normalization
K_val_linear_hog_same = Linear(X_hog_val, X_hog_val)
K_val_poly_hog_same = Poly(X_hog_val, X_hog_val)
K_val_rbf_hog_same = RBF(X_hog_val, X_hog_val)
K_val_chi_hog_same = Chi2(X_hog_val, X_hog_val, block_size=128)

In [4]:
def normalize_kernel(K):
    d = np.sqrt(np.diag(K))
    return K / (d[:,None] * d[None,:] + 1e-12)

def normalize_kernel_val(K, K_val, K_train):
    d_val = np.sqrt(np.diag(K_val))
    d_train = np.sqrt(np.diag(K_train))
    return K / (d_val[:,None] * d_train[None,:] + 1e-12)

In [5]:
K_train_linear_hog = normalize_kernel(K_train_linear_hog)
K_train_poly_hog = normalize_kernel(K_train_poly_hog)
K_train_rbf_hog = normalize_kernel(K_train_rbf_hog)
K_train_chi_hog = normalize_kernel(K_train_chi_hog)
K_val_linear_hog = normalize_kernel_val(K_val_linear_hog, K_val_linear_hog_same, K_train_linear_hog)
K_val_poly_hog = normalize_kernel_val(K_val_poly_hog, K_val_poly_hog_same, K_train_poly_hog)
K_val_rbf_hog = normalize_kernel_val(K_val_rbf_hog, K_val_rbf_hog_same, K_train_rbf_hog)
K_val_chi_hog = normalize_kernel_val(K_val_chi_hog, K_val_chi_hog_same, K_train_chi_hog)

In [6]:
Kernels_train = [
    K_train_linear_hog,
    K_train_poly_hog,
    K_train_rbf_hog,
    K_train_chi_hog,
]

Kernels_val = [
    K_val_linear_hog,
    K_val_poly_hog,
    K_val_rbf_hog,
    K_val_chi_hog,
]

In [7]:

best_score = 0
best_w = None

for i in range(4):

    w = np.zeros(4)
    w[i] = 1

    K_train = Kernels_train[i]
    K_val   = Kernels_val[i]

    model = SVC(kernel='precomputed', C=4)
    model.fit(K_train, Y_train)
    Y_pred_train = model.predict(K_train)
    train_acc = accuracy(Y_train, Y_pred_train, percentage=True)

    Y_pred_val = model.predict(K_val)
    val_acc = accuracy(Ytr_val, Y_pred_val, percentage=True)

    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Accuracy: {val_acc:.2f}%")

    if val_acc > best_score:
        best_score = val_acc
        best_w = w


Train Accuracy: 56.01%
Validation Accuracy: 16.90%
Train Accuracy: 80.76%
Validation Accuracy: 19.50%
Train Accuracy: 100.00%
Validation Accuracy: 56.50%
Train Accuracy: 100.00%
Validation Accuracy: 57.70%


In [9]:
for alpha in np.arange(0,1,0.05):

    w = np.zeros(4)   
    w[3] = alpha
    w[2] = 1-alpha

    K_train = sum(w[i] * Kernels_train[i] for i in range(4))
    K_val   = sum(w[i] * Kernels_val[i]   for i in range(4))

    model = SVC(kernel='precomputed', C=4)
    model.fit(K_train, Y_train)

    Y_pred_train = model.predict(K_train)
    train_acc = accuracy(Y_train, Y_pred_train, percentage=True)

    Y_pred_val = model.predict(K_val)
    val_acc = accuracy(Ytr_val, Y_pred_val, percentage=True)

    print(f"Train Accuracy: {train_acc:.2f}%")
    print(f"Validation Accuracy: {val_acc:.2f}%")

    if val_acc > best_score:
        best_score = val_acc
        best_w = w

print(best_score, best_w)

Train Accuracy: 100.00%
Validation Accuracy: 56.50%
Train Accuracy: 100.00%
Validation Accuracy: 56.70%
Train Accuracy: 100.00%
Validation Accuracy: 56.80%
Train Accuracy: 100.00%
Validation Accuracy: 56.80%
Train Accuracy: 100.00%
Validation Accuracy: 57.30%
Train Accuracy: 100.00%
Validation Accuracy: 57.00%
Train Accuracy: 100.00%
Validation Accuracy: 57.00%
Train Accuracy: 100.00%
Validation Accuracy: 56.90%
Train Accuracy: 100.00%
Validation Accuracy: 57.20%
Train Accuracy: 100.00%
Validation Accuracy: 57.20%
Train Accuracy: 100.00%
Validation Accuracy: 57.40%
Train Accuracy: 100.00%
Validation Accuracy: 57.80%
Train Accuracy: 100.00%
Validation Accuracy: 57.60%
Train Accuracy: 100.00%
Validation Accuracy: 57.90%
Train Accuracy: 100.00%
Validation Accuracy: 58.00%
Train Accuracy: 100.00%
Validation Accuracy: 58.10%
Train Accuracy: 100.00%
Validation Accuracy: 58.00%
Train Accuracy: 100.00%
Validation Accuracy: 58.00%
Train Accuracy: 100.00%
Validation Accuracy: 57.90%
Train Accura